# Module 6 · Lesson 01: Open-Source LLMs with Ollama

**Open-source LLMs** run locally on your machine — no API keys, no costs, full privacy.
We use **Ollama** to run models like Llama 3, Mistral, and Phi-3.

## What you will learn
1. **Ollama** setup and model management
2. Running **inference locally**
3. OpenAI-compatible API (drop-in replacement!)
4. Comparing **local vs cloud** models
5. When to use open-source vs commercial

---
> **Prerequisites:** Install Ollama from [ollama.com](https://ollama.com)
> then run `ollama pull llama3.2` or `ollama pull gemma3` to download a model.

```bash
# Download a model (run in terminal)
ollama pull gemma3

Κατέβασμα μοντέλου
# ollama pull gemma3

# Εκτέλεση μοντέλου
ollama run gemma3

# Λίστα εγκατεστημένων μοντέλων
ollama ls

# Διαγραφή μοντέλου
ollama rm gemma3

# Πληροφορίες για μοντέλο
ollama show gemma3

http://localhost:11434/api/tags

```


In [13]:
# ollama pull llama3.2
# http://localhost:11434/api/tags

import os, json, time
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv(Path.cwd().parent / ".env")

# Check if Ollama is running
try:
    import requests
    r = requests.get("http://localhost:11434/api/tags")
    models = r.json().get("models", [])
    print(f"✅ Ollama is running with {len(models)} model(s):")
    for m in models:
        size = m.get('size', 0) / 1e9
        print(f"   • {m['name']} ({size:.1f} GB)")
except:
    print("⚠️ Ollama not running. Start it with `ollama serve`")
    print("   Install from https://ollama.com")

✅ Ollama is running with 1 model(s):
   • gemma3:latest (3.3 GB)


---
## 1. Basic Inference with Ollama

Ollama provides a REST API at `localhost:11434`:

In [7]:
# Direct Ollama API
import requests
from IPython.display import display, Markdown

def ollama_chat(prompt: str, model: str = "gemma3") -> str:
    """Chat with a local model via Ollama."""
    r = requests.post("http://localhost:11434/api/chat", json={
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False
    })
    return r.json()["message"]["content"]

try:
    start = time.perf_counter()
    question = "What is Python? Answer in one sentence."
    result = ollama_chat(question)
    ms = (time.perf_counter() - start) * 1000
    display(Markdown(f"Response: **{ms:.0f}** ms"))
    display(Markdown(f"**Question:** {question}"))
    display(Markdown(f"**Answer:** {result}"))
except Exception as e:
    print(f"Error: {e}\n   Make sure Ollama is running with a model pulled.")

Response: **2876** ms

**Question:** What is Python? Answer in one sentence.

**Answer:** Python is a versatile, high-level programming language known for its readability and wide range of applications, from web development to data science.

---
## 2. OpenAI-Compatible API

Ollama supports the **OpenAI API format** — your existing code works with minimal changes!

In [10]:
# OpenAI-compatible mode
from openai import OpenAI

# Point OpenAI client at Ollama
local_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # Required but unused
)

try:
    response = local_client.chat.completions.create(
        # model="llama3.2",
        model = "gemma3",
        messages=[{"role": "user", "content": "Explain what an API is in one sentence."}],
        max_tokens=100
    )
    print(f"Local model says: {response.choices[0].message.content}")
    print("\n💡 Same OpenAI SDK, just different base_url!")
except Exception as e:
    print(f"⚠️ {e}")

Local model says: An API (Application Programming Interface) is a set of rules and specifications that allow different software applications to communicate and exchange data with each other.

💡 Same OpenAI SDK, just different base_url!


---
## 3. Local vs Cloud Comparison

| Factor | Local (Ollama) | Cloud (OpenAI) |
|--------|----------------|----------------|
| **Cost** | Free (hardware only) | Pay per token |
| **Privacy** | Data stays local | Sent to API |
| **Speed** | Depends on GPU | Very fast |
| **Quality** | Good (Llama3, Mistral) | Best (GPT-4o) |
| **Reliability** | No downtime | Rate limits possible |
| **Setup** | Install Ollama + model | API key only |

In [12]:
# Compare response quality
prompt = "Explain recursion in programming. Keep it under 50 words."

# Cloud
cloud_client = OpenAI()
cloud_start = time.perf_counter()
cloud_r = cloud_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=100
)
cloud_ms = (time.perf_counter() - cloud_start) * 1000

# Local  
try:
    local_start = time.perf_counter()
    local_r = local_client.chat.completions.create(
        model="gemma3",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=100
    )
    local_ms = (time.perf_counter() - local_start) * 1000
    
    print(f"☁️ GPT-4o-mini ({cloud_ms:.0f} ms):")
    print(f"  {cloud_r.choices[0].message.content}\n")
    print(f"🏠 Llama 3.2 ({local_ms:.0f} ms):")
    print(f"  {local_r.choices[0].message.content}")
except Exception as e:
    print(f"☁️ GPT-4o-mini ({cloud_ms:.0f} ms):")
    print(f"  {cloud_r.choices[0].message.content}")
    print(f"\n⚠️ Local model not available: {e}")

☁️ GPT-4o-mini (1604 ms):
  Recursion in programming is a technique where a function calls itself to solve smaller instances of the same problem. It typically requires a base case to terminate the recursive calls and prevent infinite loops, making it useful for tasks like calculating factorials or navigating tree structures.

🏠 Llama 3.2 (3083 ms):
  Recursion is a programming technique where a function calls itself within its own definition to solve a smaller subproblem of the same type. It continues until a base case is reached, stopping the chain of calls and returning a final value.


---
## 4. Vision Models: Seeing with Local LLMs

Open-source LLMs aren't just for text! **LLaVA** (Large Language and Vision Assistant)
can analyze images locally — no cloud API needed.

This pattern comes from a community project that built an **image description assistant
for visually impaired users** using local LLaVA models.

```bash
# Download a vision model (run in terminal)
ollama pull llava:7b-v1.6
```

In [17]:
# Step 1: Encode an image for the vision model
# ollama pull llava:7b-v1.6
import base64
from pathlib import Path

def encode_image(image_path: str) -> str:
    """Encode an image file to base64 string."""
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

# Create a simple test image using Python
try:
    from PIL import Image, ImageDraw, ImageFont
    img = Image.new('RGB', (400, 300), color='skyblue')
    draw = ImageDraw.Draw(img)
    # Draw simple shapes
    draw.rectangle([50, 100, 150, 250], fill='green')    # tree
    draw.ellipse([80, 50, 120, 100], fill='darkgreen')   # tree top
    draw.rectangle([200, 150, 350, 250], fill='red')     # house
    draw.polygon([(200, 150), (275, 80), (350, 150)], fill='brown')  # roof
    draw.ellipse([320, 20, 380, 80], fill='yellow')      # sun
    test_img_path = Path.cwd() / "demo_img.png"
    img.save(test_img_path)
    print(f"Created test image: {test_img_path}")
    b64 = encode_image(str(test_img_path))
    print(f"Base64 length: {len(b64)} chars")
except ImportError:
    print("Install Pillow for image generation: pip install Pillow")
    print("You can also use any existing image file.")
    test_img_path = None
    b64 = None

Created test image: c:\Users\user\Desktop\ai_for_devs\module_06_open_source\demo_img.png
Base64 length: 2712 chars


In [ ]:
#  Step 2: Send image to local vision model
import requests

def describe_image_local(image_path: str, prompt: str = "Describe this image in detail. Use 3-5 bullets.", 
                        model: str = "llava:7b-v1.6") -> str:
    """Use a local LLaVA model to describe an image."""
    b64_img = encode_image(image_path)

    response = requests.post("http://localhost:11434/api/generate", json={
        "model": model,
        "prompt": prompt,
        "images": [b64_img],
        "stream": False
    })
    return response.json()["response"]

# Try it!
if test_img_path and test_img_path.exists():
    try:
        start = time.perf_counter()
        description = describe_image_local(str('my-puppy.jfif'))
        ms = (time.perf_counter() - start) * 1000
        print(f"Vision model description ({ms:.0f} ms):")
        display(Markdown(description))
    except Exception as e:
        print(f"Vision model not available: {e}")
        print("Make sure to run: ollama pull llava:7b-v1.6")
        print("\nThe pattern would work like this:")
        print("  1. Encode image to base64")
        print("  2. Send to Ollama with 'images' parameter")
        print("  3. Get text description back")
else:
    print("No test image available. Install Pillow or provide an image path.")
    print("The pattern: encode_image() -> ollama_generate(images=[b64]) -> text")

Vision model description (5535 ms):


 - This is a color photograph featuring a golden retriever dog in mid-air, seemingly captured while it's playing with a ball on a beach during sunset or sunrise, as suggested by the warm hues of the sky and the calm sea in the background.

- The dog appears to be brownish gold with a fluffy coat, holding a tennis ball in its mouth. It is positioned centrally in the image, drawing attention due to its dynamic pose.

- The beach scene includes waves lapping at the shore, wet sand, and the horizon line where the sun or moon is partially visible, casting a soft glow on the scene. The ocean water reflects some of the sky's colors.

- There are no texts present in the image, focusing entirely on the natural scene and the dog's action.

- The photograph captures an idyllic moment, conveying joy and playfulness through the dog's motion and the serene coastal environment. 

In [21]:
#  Step 3: Accessibility assistant (real-world use case)

def accessibility_describe(image_path: str) -> str:
    """Describe an image for visually impaired users."""
    prompt = """You are an accessibility assistant helping visually impaired users.
Describe this image in detail, focusing on:
1. Main objects and their positions
2. Colors and lighting
3. Any text visible in the image
4. The overall scene and mood

Be descriptive but concise. Use spatial language (left, right, center, etc.)."""

    try:
        return describe_image_local(image_path, prompt)
    except Exception:
        # Fallback: use cloud vision model if local unavailable
        cloud = OpenAI()
        b64_img = encode_image(image_path)
        r = cloud.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {
                        "url": f"data:image/png;base64,{b64_img}"
                    }}
                ]
            }],
            max_tokens=300
        )
        return "[Cloud fallback] " + r.choices[0].message.content

# Test the accessibility assistant
if test_img_path and test_img_path.exists():
    print("Accessibility description:")
    desc = accessibility_describe(str('my-puppy.jfif'))
    display(Markdown(desc))
else:
    print("Provide an image path to test the accessibility assistant.")

Accessibility description:


 1. Main objects and their positions:
- Golden retriever dog, centered, in mid-air jump with a sports ball in its mouth.
- Sunset over the ocean with vibrant colors in the sky.
- Beach scene with waves near shoreline where dog is jumping.

2. Colors and lighting:
- Warm sunset with hues of orange, pink, and blue dominating the upper part of the image.
- Golden yellow sand on beach contrasting with the ocean's cooler tones.

3. Text visible in the image:
- None.

4. Overall scene and mood:
The image captures a joyful moment of a dog leaping into the air, likely chasing a ball or just enjoying the freedom of running on the beach. The sunset creates a serene backdrop to this active scene, adding a sense of tranquility despite the energy of the jumping dog. The playful and carefree mood is conveyed through the dog's action, the open space of the beach, and the beautiful colors of the sky. 

> **Note:** The `accessibility_describe()` function demonstrates the **fallback pattern** from Module 3:
> try local LLaVA first, fall back to cloud GPT-4o-mini if Ollama isn't available.

> **Exercise:** Extend `describe_image_local()` to stream the response token-by-token
> (set `"stream": True` and iterate over the response). This gives faster perceived responses
> for large image descriptions.

---
## 5. Hybrid Architecture: Cloud + Local Fallback

Production systems should work even when infrastructure is degraded.
This pattern tries the local model first, falls back to cloud if Ollama is offline.

In [24]:
# Smart completion: local-first with cloud fallback

class HybridLLM:
    """Try local Ollama first, fall back to OpenAI cloud."""

    def __init__(self):
        self.cloud = OpenAI()
        self.local = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
        self.stats = {"local": 0, "cloud": 0}

    def complete(self, prompt: str, prefer_local: bool = True) -> dict:
        """Complete with automatic fallback."""
        if prefer_local:
            try:
                start = time.perf_counter()
                r = self.local.chat.completions.create(
                    # model="llama3.2",
                    model="gemma3",
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=200, timeout=15
                )
                ms = (time.perf_counter() - start) * 1000
                self.stats["local"] += 1
                return {"text": r.choices[0].message.content,
                        "backend": "local (Ollama)", "latency_ms": round(ms), "cost": 0.0}
            except Exception:
                pass  # Fall through to cloud

        start = time.perf_counter()
        r = self.cloud.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200
        )
        ms = (time.perf_counter() - start) * 1000
        cost = (r.usage.prompt_tokens + r.usage.completion_tokens) / 1e6 * 0.6
        self.stats["cloud"] += 1
        return {"text": r.choices[0].message.content,
                "backend": "cloud (OpenAI)", "latency_ms": round(ms), "cost": cost}

# Test it
hybrid = HybridLLM()
result = hybrid.complete("What is Python? One sentence.")

print(f"Response: {result['text']}")
print(f"Backend: {result['backend']}")
print(f"Latency: {result['latency_ms']} ms | Cost: ${result['cost']:.6f}")
print(f"\nUsage stats: {hybrid.stats}")

Response: Python is a versatile, high-level programming language known for its readability and wide range of applications, from web development to data science.
Backend: local (Ollama)
Latency: 5804 ms | Cost: $0.000000

Usage stats: {'local': 1, 'cloud': 0}


> **Exercise:** Modify `HybridLLM` to add a latency threshold — if the local model
> takes more than 5 seconds, automatically switch to cloud for the next request.
> Track and display which backend was chosen for each call.

---
## Key Takeaways 📝

| Concept | Detail |
|---------|--------|
| **Ollama** | Run open-source LLMs locally with one command |
| **OpenAI compatibility** | Change `base_url` — same code works |
| **Use local for** | Privacy-sensitive data, offline, cost savings |
| **Use cloud for** | Best quality, speed, production scale |
| **Hybrid approach** | Local for development, cloud for production |
| **Vision models** | LLaVA can analyze images locally — no cloud needed |
| **Accessibility** | Real social impact: image descriptions for visually impaired |

---
**Explore:** Check `ollama_quickstart/` for more examples